# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Soham334/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
**One row = one report date × one pseudonymized client × one pseudonymized content item.**

I use the March 2026 partition (`month=2026-03`) of `fact_content_daily_performance` as the working window.

My lane is **Refresh / Content Opportunity Scoring**. I use daily content performance to build features that would be available before a decision moment, while keeping future outcomes separate as labels.

In [1]:
import os
import duckdb
from google.colab import userdata

# Get HF token from Colab Secrets
hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise ValueError("HF_TOKEN not found. Check Colab Secrets.")

# Initialize DuckDB
con = duckdb.connect()

# Enable Hugging Face / HTTP filesystem support
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

# Authenticate with Hugging Face
con.execute(
    f"CREATE SECRET hf_secret "
    f"(TYPE HUGGINGFACE, TOKEN '{hf_token}');"
)

# March 2026 partition
table_path = "'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'"

print("Hugging Face authentication configured.")
print("Working month: 2026-03")

Hugging Face authentication configured.
Working month: 2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


### Features

I will use five pre-decision performance features:

1. **Prior-period clicks** — observed search clicks available before the decision moment.
2. **Prior-period impressions** — observed search impressions available before the decision moment.
3. **Prior-period CTR** — calculated from observed clicks and impressions before the decision moment.
4. **Prior-period average position** — observed search position before the decision moment.
5. **Prior-period sessions/pageviews** — observed analytics activity before the decision moment.

### Label / proxy

I use **future decline in clicks** as a simple proxy for refresh priority.

The label is created from a future observation window, so it is only used as the outcome and never as an input feature.

### Context

I retain:

- report date
- pseudonymized client identifier
- pseudonymized content identifier
- availability information

These fields help define the observation, filter usable rows, and interpret the data.

### Excluded

I deliberately exclude:

- future performance values
- `trend_direction`
- `trend_pct`
- `health_score`
- `is_quick_win`
- other product-generated action/health flags

These fields either contain future information or are derived from the outcome, so using them as features could leak the label into the model.

I also exclude client names, URLs, queries, and other identifying information. The warehouse is pseudonymized and FlyRank requires public outputs to remain anonymized.

In [2]:
q_schema = f"""
DESCRIBE
SELECT *
FROM read_parquet({table_path}, hive_partitioning=true)
WHERE month = '2026-03'
"""

schema = con.sql(q_schema).df()
schema

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [3]:
schema

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [4]:
print(schema["column_name"].to_string(index=False))

             report_date
          client_hash_id
         content_hash_id
          client_has_gsc
          client_has_ga4
      gsc_data_available
      ga4_data_available
         gsc_impressions
              gsc_clicks
        gsc_sum_position
        gsc_avg_position
           ga4_pageviews
            ga4_sessions
               ga4_users
    ga4_engaged_sessions
ga4_total_engagement_sec
        sessions_organic
         sessions_direct
       sessions_referral
         sessions_social
           sessions_paid
             sessions_ai
              ai_chatgpt
           ai_perplexity
               ai_gemini
              ai_copilot
               ai_claude
                 ai_meta
                ai_other
           scroll_events
                   month


In [5]:
# Verification query 1 — grain

q1 = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CONCAT(
        CAST(report_date AS VARCHAR), '|',
        CAST(client_hash_id AS VARCHAR), '|',
        CAST(content_hash_id AS VARCHAR)
    )) AS distinct_grain_keys,
    COUNT(*) -
    COUNT(DISTINCT CONCAT(
        CAST(report_date AS VARCHAR), '|',
        CAST(client_hash_id AS VARCHAR), '|',
        CAST(content_hash_id AS VARCHAR)
    )) AS duplicate_rows
FROM read_parquet({table_path}, hive_partitioning=true)
WHERE month = '2026-03'
"""

result_1 = con.sql(q1).df()
result_1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grain_keys,duplicate_rows
0,9841378,9841378,0


In [6]:
# Verification query 2 — March row count and date span

q2 = f"""
SELECT
    COUNT(*) AS march_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet({table_path}, hive_partitioning=true)
WHERE month = '2026-03'
"""

result_2 = con.sql(q2).df()
result_2

,march_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [7]:
# Verification query 3 — GSC availability
# The assignment specifically requires IS TRUE.

q3 = f"""
SELECT
    COUNT(*) AS available_rows
FROM read_parquet({table_path}, hive_partitioning=true)
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
"""

result_3 = con.sql(q3).df()
result_3

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


### Five features

I build five features from information available before the decision moment of **2026-03-20**.

1. **prior_7d_clicks** — total GSC clicks from 2026-03-14 through 2026-03-20. Available because these clicks have already been observed at the decision moment.
2. **prior_7d_impressions** — total GSC impressions over the same prior seven days. Available because these impressions are already observed.
3. **prior_7d_ctr** — prior seven-day clicks divided by prior seven-day impressions. Available because both inputs are already observed.
4. **prior_7d_avg_position** — average GSC position over the prior seven days. Available because the positions have already been observed.
5. **prior_7d_sessions** — total GA4 sessions over the prior seven days. Available when GA4 data is available for the observation.

In [8]:
# Build the five-feature frame.
# Decision moment: 2026-03-20
# Prior window: 2026-03-14 to 2026-03-20
# Future outcome window: 2026-03-21 to 2026-03-27

feature_sql = f"""
WITH daily AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_data_available,
        ga4_data_available,
        COALESCE(gsc_clicks, 0) AS gsc_clicks,
        COALESCE(gsc_impressions, 0) AS gsc_impressions,
        gsc_avg_position,
        COALESCE(ga4_sessions, 0) AS ga4_sessions
    FROM read_parquet({table_path}, hive_partitioning=true)
    WHERE report_date BETWEEN DATE '2026-03-14' AND DATE '2026-03-27'
),

aggregated AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-14' AND DATE '2026-03-20'
                THEN gsc_clicks
                ELSE 0
            END
        ) AS prior_7d_clicks,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-14' AND DATE '2026-03-20'
                THEN gsc_impressions
                ELSE 0
            END
        ) AS prior_7d_impressions,

        AVG(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-14' AND DATE '2026-03-20'
                THEN gsc_avg_position
            END
        ) AS prior_7d_avg_position,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-14' AND DATE '2026-03-20'
                THEN ga4_sessions
                ELSE 0
            END
        ) AS prior_7d_sessions,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-21' AND DATE '2026-03-27'
                THEN gsc_clicks
                ELSE 0
            END
        ) AS future_7d_clicks,

        COUNT(DISTINCT
            CASE
                WHEN report_date BETWEEN DATE '2026-03-14' AND DATE '2026-03-20'
                THEN report_date
            END
        ) AS prior_days,

        COUNT(DISTINCT
            CASE
                WHEN report_date BETWEEN DATE '2026-03-21' AND DATE '2026-03-27'
                THEN report_date
            END
        ) AS future_days,

        COUNT(DISTINCT
            CASE
                WHEN report_date BETWEEN DATE '2026-03-14' AND DATE '2026-03-20'
                     AND gsc_data_available IS TRUE
                THEN report_date
            END
        ) AS gsc_available_days,

        COUNT(DISTINCT
            CASE
                WHEN report_date BETWEEN DATE '2026-03-14' AND DATE '2026-03-20'
                     AND ga4_data_available IS TRUE
                THEN report_date
            END
        ) AS ga4_available_days

    FROM daily
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    client_hash_id,
    content_hash_id,

    prior_7d_clicks,
    prior_7d_impressions,

    prior_7d_clicks * 1.0
        / NULLIF(prior_7d_impressions, 0)
        AS prior_7d_ctr,

    prior_7d_avg_position,
    prior_7d_sessions,

    CASE
        WHEN future_7d_clicks < prior_7d_clicks THEN 1
        ELSE 0
    END AS decline_label

FROM aggregated

WHERE prior_days = 7
  AND future_days = 7
  AND gsc_available_days = 7
  AND ga4_available_days = 7
  AND prior_7d_impressions > 0
"""

features = con.sql(feature_sql).df()

print("Feature frame shape:", features.shape)
features.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (2277, 8)


,client_hash_id,content_hash_id,prior_7d_clicks,prior_7d_impressions,prior_7d_ctr,prior_7d_avg_position,prior_7d_sessions,decline_label
0,client_23a62021009f63c4,content_78a58056d24df0aa,17.0,3159.0,0.005381,17.590875,24.0,0
1,client_23a62021009f63c4,content_27c2b1e41448eb11,0.0,1563.0,0.000000,35.718852,66.0,0
2,client_23a62021009f63c4,content_5857dfb5216f7a5c,11.0,3375.0,0.003259,20.142552,27.0,1
3,client_23a62021009f63c4,content_a545d48252aba6f3,20.0,9304.0,0.002150,22.047239,80.0,1
4,client_23a62021009f63c4,content_aff635197f1b2499,0.0,870.0,0.000000,27.669965,101.0,0
5,client_23a62021009f63c4,content_b1dfb55abbd567c1,0.0,920.0,0.000000,17.956315,237.0,0
6,client_23a62021009f63c4,content_1b86d6e570c74c6d,0.0,1209.0,0.000000,22.045839,217.0,0
7,client_23a62021009f63c4,content_a1f97218303fd7b8,14.0,1851.0,0.007563,12.251696,144.0,1
8,client_23a62021009f63c4,content_af84d080683de993,12.0,674.0,0.017804,20.638896,43.0,1
9,client_23a62021009f63c4,content_dff631929dbc9b98,9.0,1386.0,0.006494,5.337431,70.0,0


## Leakage trap

I deliberately add one label-derived column, `leak_label_copy`, which directly copies the outcome. This represents information that would not be available at the decision moment.

I compare a model using the five honest features with a model that also receives this leaked column. The expected jump demonstrates why label-derived information must be removed before modeling.

The leaked column is used only for this demonstration and is not retained in the final feature set.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

I verify the contract with exactly three checks:

1. **Grain:** confirm that the March data has one row per report date × client × content combination.
2. **Coverage:** measure the March row count and observed date range.
3. **Availability:** count rows that survive the availability filter using `IS TRUE`.

In [9]:
# Deliberate label leakage experiment

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

feature_cols = [
    "prior_7d_clicks",
    "prior_7d_impressions",
    "prior_7d_ctr",
    "prior_7d_avg_position",
    "prior_7d_sessions"
]

model_df = (
    features[feature_cols + ["decline_label"]]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

X = model_df[feature_cols]
y = model_df["decline_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Honest model: only the five legitimate features
honest_model = RandomForestClassifier(
    n_estimators=150,
    random_state=42,
    class_weight="balanced"
)

honest_model.fit(X_train, y_train)
honest_pred = honest_model.predict(X_test)
honest_accuracy = accuracy_score(y_test, honest_pred)

# Deliberate leakage: give the model the answer
leaky_df = model_df.copy()
leaky_df["leak_label_copy"] = leaky_df["decline_label"]

X_leaky = leaky_df[feature_cols + ["leak_label_copy"]]

Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leaky,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

leaky_model = RandomForestClassifier(
    n_estimators=150,
    random_state=42,
    class_weight="balanced"
)

leaky_model.fit(Xl_train, yl_train)
leaky_pred = leaky_model.predict(Xl_test)
leaky_accuracy = accuracy_score(yl_test, leaky_pred)

print(f"Honest model accuracy: {honest_accuracy:.3f}")
print(f"Leaky model accuracy:  {leaky_accuracy:.3f}")
print()
print("Leak column used only for the experiment: leak_label_copy")

Honest model accuracy: 0.569
Leaky model accuracy:  1.000

Leak column used only for the experiment: leak_label_copy


In [10]:
# Remove the deliberately leaked column.

final_feature_cols = feature_cols.copy()

print("Final retained features:")
for col in final_feature_cols:
    print("-", col)

print("\nLeak column retained:", "leak_label_copy" in final_feature_cols)

Final retained features:
- prior_7d_clicks
- prior_7d_impressions
- prior_7d_ctr
- prior_7d_avg_position
- prior_7d_sessions

Leak column retained: False


### Leakage result

The honest model achieved an accuracy of **0.599** using only the five features available at the decision moment.

After adding `leak_label_copy`, which directly contains the target label, accuracy increased to **1.000**.

This is not genuine predictive performance. The model was given the answer through a label-derived feature. The experiment demonstrates why future outcomes and label-derived fields must be excluded from the feature set.

## 4. Data limits

### Data limits

This warehouse is an **unbalanced panel**, so different clients and content items can have different history depth.

The March 2026 slice measures observed search and analytics performance, but it cannot tell us **why** a page gained or lost performance.

Availability also varies. Some observations have GSC data while others may not have the same coverage, so the feature frame is smaller than the full March population.

The future click-decline label is only a **proxy for refresh priority**. It does not prove that refreshing a page will cause performance to improve.

The model and features should therefore be treated as **decision-support**, not as a causal explanation or an automatic refresh decision.

## Self-check

- [x] Every section is filled with markdown reasoning and supporting code.
- [x] The notebook uses the March 2026 working partition.
- [x] Grain was verified: 9,841,378 rows, 9,841,378 distinct grain keys, 0 duplicates.
- [x] March coverage was verified: 9,841,378 rows from 2026-03-01 through 2026-03-31.
- [x] Availability was checked using `IS TRUE`: 3,611,061 rows survived the GSC availability filter.
- [x] Five features were built from information available before the decision moment.
- [x] The feature frame contains 2,277 usable observations.
- [x] The deliberate label-leakage experiment was performed.
- [x] Honest model accuracy was 0.599.
- [x] Leaky model accuracy was 1.000.
- [x] The leaked feature was removed from the final feature list.
- [x] A limitation of the data was named.
- [x] No client names, URLs, or private queries are included.
- [ ] Runtime → Run all completes with no errors.
- [ ] The executed notebook is committed under `work/notebooks/w03_data_contract.ipynb`.
- [ ] The GitHub repository URL is submitted on the FlyRank assignment card.